# The second best model we'll do
* bangbangbang

## O. Setup 

In [0]:
NOM_EQUIPE = "telecacaton"   # ← remplacez par le nom de votre équipe

# Ne touchez pas au reste
TABLE_PREDICTIONS = f"workspace.default.predictions_equipe_{NOM_EQUIPE}"
print(f"Votre table de prédictions : {TABLE_PREDICTIONS}")

In [0]:
%sql GRANT MODIFY ON TABLE workspace.default.predictions_equipe_telecacaton TO `cyprien.mas@telecom-paris.fr`;
GRANT MODIFY ON TABLE workspace.default.predictions_equipe_telecacaton TO `hugo.hennion@telecom-paris.fr`;
GRANT MODIFY ON TABLE workspace.default.predictions_equipe_telecacaton TO `clement.pesquet@telecom-paris.fr`;

## 1. Model

### Import et Setup

In [0]:
%pip install lightgbm
#dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
import lightgbm as lgb

### Chargement des données

In [0]:
# COMMAND ----------
train_df = spark.table("workspace.default.histo_ventes_train")
test_df  = spark.table("workspace.default.histo_ventes_test")

cols_utiles = ["semaine", "code_agence", "code_article", "quantite"]

print("⏳ Chargement train...")
train_pd = train_df.select(cols_utiles).toPandas()
print(f"✅ Train : {len(train_pd):,} lignes")

print("⏳ Chargement test...")
test_pd = test_df.select(["semaine", "code_agence", "code_article"]).toPandas()
print(f"✅ Test  : {len(test_pd):,} lignes")

def parse_semaine(df):
    df = df.copy()
    df["annee"]   = df["semaine"].str.split("-").str[0].astype(int)
    df["num_sem"] = df["semaine"].str.split("-").str[1].astype(int)
    df["week_id"] = df["annee"] * 100 + df["num_sem"]
    return df

train_pd = parse_semaine(train_pd)
test_pd  = parse_semaine(test_pd)

train_pd["is_test"] = 0
test_pd["is_test"]  = 1
test_pd["quantite"] = np.nan

all_df = pd.concat([train_pd, test_pd], ignore_index=True)
all_df = all_df.sort_values(["code_agence", "code_article", "week_id"]).reset_index(drop=True)

print(f"Total : {len(all_df):,} lignes | {all_df.memory_usage(deep=True).sum()/1e6:.0f} MB")

### 2. Feature Engineering

In [0]:
# COMMAND ----------
def add_features(df):
    df       = df.copy()
    grp      = ["code_agence", "code_article"]
    pair_key = (df["code_agence"].astype(str) + "_" + df["code_article"].astype(str))

    # ── Lags ─────────────────────────────────────────────────────────────
    for lag in [1, 2, 4, 8, 13, 26, 52, 104]:
        df[f"lag_{lag}"] = df.groupby(grp)["quantite"].shift(lag)

    # ── Rolling (shift(1) avant rolling → pas de fuite) ───────────────────
    shifted = df.groupby(grp)["quantite"].shift(1)
    for window in [4, 13, 26, 52]:
        df[f"roll_mean_{window}"] = (
            shifted.groupby(pair_key)
                   .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )
        df[f"roll_std_{window}"] = (
            shifted.groupby(pair_key)
                   .transform(lambda x: x.rolling(window, min_periods=1).std())
        )

    # ── Taux de zéros glissants ───────────────────────────────────────────
    is_zero      = (df["quantite"] == 0).astype(float)
    shifted_zero = is_zero.groupby(pair_key).shift(1)
    for window in [13, 26, 52]:
        df[f"zero_rate_{window}"] = (
            shifted_zero.groupby(pair_key)
                        .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )

    # ── Tendances ─────────────────────────────────────────────────────────
    df["trend_8"] = (
        df.groupby(grp)["quantite"].shift(1)
        - df.groupby(grp)["quantite"].shift(9)
    ) / 8
    df["trend_26"] = (
        df.groupby(grp)["quantite"].shift(1)
        - df.groupby(grp)["quantite"].shift(27)
    ) / 26

    # ── Stats globales par paire ──────────────────────────────────────────
    # ANTI-OVERFITTING : calculées sur train entier, pas juste pré-2025
    # Le modèle voit la même distribution que le test
    train_only = df[df["is_test"] == 0]

    pair_stats = (
        train_only.groupby(grp)["quantite"]
        .agg(
            pair_mean="mean",
            pair_median="median",
            pair_max="max",
            pair_count="count",
            pair_std="std",
        )
        .reset_index()
    )
    df = df.merge(pair_stats, on=grp, how="left")

    # ── Stats semaine × paire ─────────────────────────────────────────────
    sem_stats = (
        train_only.groupby(grp + ["num_sem"])["quantite"]
        .agg(sem_mean="mean", sem_max="max", sem_median="median")
        .reset_index()
    )
    df = df.merge(sem_stats, on=grp + ["num_sem"], how="left")

    # ── Stats agence & article ────────────────────────────────────────────
    agence_stats = (
        train_only.groupby("code_agence")["quantite"]
        .agg(agence_mean="mean", agence_median="median")
        .reset_index()
    )
    article_stats = (
        train_only.groupby("code_article")["quantite"]
        .agg(article_mean="mean", article_median="median")
        .reset_index()
    )
    df = df.merge(agence_stats,  on="code_agence",  how="left")
    df = df.merge(article_stats, on="code_article", how="left")

    # ── Semaines actives ──────────────────────────────────────────────────
    active_weeks = (
        train_only.groupby(grp)
        .apply(lambda x: (x["quantite"] > 0).sum())
        .reset_index()
        .rename(columns={0: "n_active_weeks"})
    )
    df = df.merge(active_weeks, on=grp, how="left")
    df["pct_active"]      = df["n_active_weeks"] / (df["pair_count"] + 1e-5)
    df["cv_pair"]         = df["pair_std"] / (df["pair_mean"] + 1e-5)
    df["ratio_sem_vs_pair"] = df["sem_mean"] / (df["pair_mean"] + 1e-5)
    df["ratio_n1_vs_mean"]  = df["lag_52"]  / (df["pair_mean"] + 1e-5)

    # ── Zero rate global ──────────────────────────────────────────────────
    zero_rate_global = (
        train_only.groupby(grp)["quantite"]
        .apply(lambda x: (x == 0).mean())
        .reset_index()
        .rename(columns={"quantite": "zero_rate_global"})
    )
    df = df.merge(zero_rate_global, on=grp, how="left")

    # ── Calendaire ────────────────────────────────────────────────────────
    vacances = [1, 2, 7, 8, 17, 18, 19, 28, 29, 30, 31, 32, 43, 44, 52]
    df["is_vacances"]      = df["num_sem"].isin(vacances).astype(int)
    df["is_fin_trimestre"] = df["num_sem"].isin([13, 26, 39, 52]).astype(int)
    df["mois"]             = ((df["num_sem"] - 1) // 4 + 1).clip(1, 12)

    # ── Saisonnalité ──────────────────────────────────────────────────────
    df["sin_sem"]  = np.sin(2 * np.pi * df["num_sem"] / 52)
    df["cos_sem"]  = np.cos(2 * np.pi * df["num_sem"] / 52)
    df["sin_mois"] = np.sin(2 * np.pi * df["mois"]    / 12)
    df["cos_mois"] = np.cos(2 * np.pi * df["mois"]    / 12)

    return df

print("⏳ Feature engineering...")
all_df = add_features(all_df)
print(f"✅ Shape : {all_df.shape}")

### 3. Split Train / Validation / Test

In [0]:
# COMMAND ----------
FEATURES = [
    "lag_1","lag_2","lag_4","lag_8","lag_13","lag_26","lag_52","lag_104",
    "roll_mean_4","roll_mean_13","roll_mean_26","roll_mean_52",
    "roll_std_4", "roll_std_13", "roll_std_26", "roll_std_52",
    "zero_rate_13","zero_rate_26","zero_rate_52","zero_rate_global",
    "trend_8","trend_26","ratio_n1_vs_mean","ratio_sem_vs_pair","cv_pair",
    "pair_mean","pair_median","pair_max","pair_count","pair_std",
    "sem_mean","sem_max","sem_median",
    "agence_mean","agence_median",
    "article_mean","article_median",
    "n_active_weeks","pct_active",
    "is_vacances","is_fin_trimestre","mois","num_sem","annee",
    "sin_sem","cos_sem","sin_mois","cos_mois",
]
FEATURES = [f for f in FEATURES if f in all_df.columns]
print(f"Nombre de features : {len(FEATURES)}")

# ANTI-OVERFITTING : validation sur 2024 (pas 2025)
# → le modèle est évalué sur une période similaire au test
# → évite que le modèle mémorise 2025 pendant l'entraînement
train_mask = (all_df["is_test"] == 0) & (all_df["semaine"] < "2024-01")
val_mask   = (all_df["is_test"] == 0) & (all_df["semaine"].between("2024-01", "2024-52"))
test_mask  =  all_df["is_test"] == 1

X_train = all_df.loc[train_mask, FEATURES]
y_train = all_df.loc[train_mask, "quantite"]
X_val   = all_df.loc[val_mask,   FEATURES]
y_val   = all_df.loc[val_mask,   "quantite"]
X_test  = all_df.loc[test_mask,  FEATURES]

print(f"X_train : {X_train.shape} | X_val : {X_val.shape} | X_test : {X_test.shape}")

### 4. Entraînement LightGBM

In [0]:
# COMMAND ----------
def wape_eval(y_pred, dataset):
    y_true = dataset.get_label()
    return "wape", np.sum(np.abs(y_pred - y_true)) / (np.sum(y_true) + 1e-10), False

def wape_score(pred, true):
    return np.sum(np.abs(pred - true)) / (np.sum(true) + 1e-10)

dtrain = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
dval   = lgb.Dataset(X_val,   label=y_val,   free_raw_data=False, reference=dtrain)

BASE = {
    "metric":            "None",
    "n_jobs":            -1,
    "seed":              42,
    "verbose":           -1,
    "num_leaves":        255,
    "min_child_samples": 20,
    "feature_fraction":  0.7,
    "bagging_fraction":  0.7,
    "bagging_freq":      1,
    "reg_alpha":         0.05,
    "reg_lambda":        1.0,
    "min_gain_to_split": 0.001,
}

# ── Tuning power + num_leaves simultané ───────────────────────────────────
# On teste plus de combinaisons pour trouver le vrai optimum
print("⚙️  Tuning tweedie_variance_power + num_leaves...")
best_power, best_leaves, best_wape_pow = 1.2, 255, float("inf")

for power in [1.2, 1.5, 1.7]:
    for leaves in [127, 255, 511]:
        m = lgb.train(
            {**BASE, "objective": "tweedie",
             "tweedie_variance_power": power,
             "num_leaves": leaves,
             "learning_rate": 0.05},
            dtrain,
            num_boost_round = 1000,
            valid_sets      = [dval],
            valid_names     = ["val"],
            feval           = wape_eval,
            callbacks       = [lgb.early_stopping(50, min_delta=1e-4),
                               lgb.log_evaluation(1000)],
        )
        w = wape_score(m.predict(X_val), y_val.values)
        print(f"  power={power} leaves={leaves} → WAPE={w:.4f}")
        if w < best_wape_pow:
            best_wape_pow, best_power, best_leaves = w, power, leaves

print(f"✅ Meilleur power={best_power} leaves={best_leaves} (WAPE={best_wape_pow:.4f})")
PARAMS = {**BASE, "objective": "tweedie",
          "tweedie_variance_power": best_power,
          "num_leaves": best_leaves}

# ── Phase 1 : exploration large ───────────────────────────────────────────
print("\n🔍 Phase 1 — LR=0.05...")
model_p1 = lgb.train(
    {**PARAMS, "learning_rate": 0.05},
    dtrain,
    num_boost_round = 2000,
    valid_sets      = [dtrain, dval],
    valid_names     = ["train", "val"],
    feval           = wape_eval,
    callbacks       = [lgb.early_stopping(60, min_delta=1e-4),
                       lgb.log_evaluation(100)],
)
print(f"✅ iter={model_p1.best_iteration} | WAPE val={wape_score(model_p1.predict(X_val), y_val.values):.4f}")

# ── Phase 2 : affinage moyen ──────────────────────────────────────────────
print("\n🎯 Phase 2 — LR=0.01...")
model_p2 = lgb.train(
    {**PARAMS, "learning_rate": 0.01},
    dtrain,
    num_boost_round = 3000,
    valid_sets      = [dtrain, dval],
    valid_names     = ["train", "val"],
    feval           = wape_eval,
    init_model      = model_p1,
    callbacks       = [lgb.early_stopping(80, min_delta=5e-5),
                       lgb.log_evaluation(100)],
)
print(f"✅ iter={model_p2.best_iteration} | WAPE val={wape_score(model_p2.predict(X_val), y_val.values):.4f}")

# ── Phase 3 : affinage fin — plus de capacité, moins de régularisation ────
print("\n🏁 Phase 3 — LR=0.005...")
model_p3 = lgb.train(
    {**PARAMS, "learning_rate": 0.005,
     "num_leaves":        max(best_leaves, 511),  # plus de capacité
     "min_child_samples": 10,
     "feature_fraction":  0.6,   # plus de diversité
     "bagging_fraction":  0.6,
     "reg_alpha":         0.01,
     "reg_lambda":        0.3,
     "min_gain_to_split": 0.0001,
    },
    dtrain,
    num_boost_round = 4000,
    valid_sets      = [dtrain, dval],
    valid_names     = ["train", "val"],
    feval           = wape_eval,
    init_model      = model_p2,
    callbacks       = [lgb.early_stopping(120, min_delta=1e-5),
                       lgb.log_evaluation(100)],
)
print(f"✅ iter={model_p3.best_iteration} | WAPE val={wape_score(model_p3.predict(X_val), y_val.values):.4f}")

# ── Phase 4 : micro-affinage ultra-fin ────────────────────────────────────
print("\n✨ Phase 4 — LR=0.001 (micro-affinage)...")
model = lgb.train(
    {**PARAMS, "learning_rate": 0.001,
     "num_leaves":        max(best_leaves, 511),
     "min_child_samples": 5,
     "feature_fraction":  0.5,
     "bagging_fraction":  0.5,
     "reg_alpha":         0.005,
     "reg_lambda":        0.1,
     "min_gain_to_split": 0.00001,
    },
    dtrain,
    num_boost_round = 5000,
    valid_sets      = [dtrain, dval],
    valid_names     = ["train", "val"],
    feval           = wape_eval,
    init_model      = model_p3,
    callbacks       = [lgb.early_stopping(150, min_delta=1e-6),
                       lgb.log_evaluation(200)],
)
print(f"✅ iter={model.best_iteration} | WAPE val={wape_score(model.predict(X_val), y_val.values):.4f}")

# ── Choisir le meilleur modèle entre les 4 phases ─────────────────────────
candidates = {
    "phase1": model_p1,
    "phase2": model_p2,
    "phase3": model_p3,
    "phase4": model,
}
best_name, best_model, best_wape_final = "", None, float("inf")
print("\n📊 Comparaison des phases :")
for name, m in candidates.items():
    w = wape_score(m.predict(X_val), y_val.values)
    print(f"  {name} → WAPE={w:.4f}")
    if w < best_wape_final:
        best_wape_final, best_name, best_model = w, name, m

model = best_model
print(f"\n🏆 Meilleur modèle : {best_name} → WAPE={best_wape_final:.4f}")

# ── Diagnostic overfitting ────────────────────────────────────────────────
wape_train = wape_score(model.predict(X_train), y_train.values)
wape_val_  = wape_score(model.predict(X_val),   y_val.values)
gap        = wape_val_ - wape_train
print(f"\nWAPE train : {wape_train:.4f}")
print(f"WAPE val   : {wape_val_:.4f}")
print(f"Gap        : {gap:.4f} {'⚠️ overfit' if gap > 0.05 else '✅ OK'}")

### 5. Validation & Score

In [0]:
# COMMAND ----------
val_preds_raw = model.predict(X_val, num_iteration=model.best_iteration)
val_preds     = np.clip(np.round(val_preds_raw), 0, None).astype(int)
print(f"WAPE LightGBM brut : {wape_score(val_preds, y_val.values):.4f}")

# ── Post-processing zéros ─────────────────────────────────────────────────
pair_zero_rate = (
    train_pd.groupby(["code_agence", "code_article"])["quantite"]
    .apply(lambda x: (x == 0).mean())
    .reset_index()
    .rename(columns={"quantite": "zero_rate_global"})
)

val_rows = all_df.loc[val_mask, ["semaine", "code_agence", "code_article"]].copy()
val_rows["quantite"] = val_preds
val_rows = val_rows.merge(pair_zero_rate, on=["code_agence", "code_article"], how="left")
val_rows["quantite"] = np.where(val_rows["zero_rate_global"] > 0.98, 0, val_rows["quantite"])
print(f"WAPE après filtre zéros    : {wape_score(val_rows['quantite'].values, y_val.values):.4f}")

# ── Blend N-1 via join Pandas (rapide) ───────────────────────────────────
n1_lookup = (
    train_pd[["semaine", "code_agence", "code_article", "quantite"]]
    .assign(
        annee   = lambda d: d["semaine"].str.split("-").str[0].astype(int),
        num_sem = lambda d: d["semaine"].str.split("-").str[1],
    )
    .assign(semaine_join = lambda d: (d["annee"] + 1).astype(str) + "-" + d["num_sem"])
    .rename(columns={"semaine_join": "semaine_target", "quantite": "quantite_n1"})
    [["semaine_target", "code_agence", "code_article", "quantite_n1"]]
)

val_rows = val_rows.rename(columns={"semaine": "semaine_target"})
val_rows = val_rows.merge(
    n1_lookup, on=["semaine_target", "code_agence", "code_article"], how="left"
)
val_rows = val_rows.rename(columns={"semaine_target": "semaine"})

# Blend appris sur validation : Ridge
from sklearn.linear_model import Ridge
import numpy as np

val_lgbm = val_rows["quantite"].values.astype(float)
val_n1   = val_rows["quantite_n1"].fillna(0).values.astype(float)

# Moyenne historique
pair_mean_lookup = train_pd.groupby(["code_agence","code_article"])["quantite"].mean().reset_index().rename(columns={"quantite":"pair_mean_bl"})
val_rows2 = all_df.loc[val_mask, ["semaine","code_agence","code_article"]].copy()
val_rows2 = val_rows2.merge(pair_mean_lookup, on=["code_agence","code_article"], how="left")
val_mean  = val_rows2["pair_mean_bl"].fillna(0).values.astype(float)

preds_stack = np.column_stack([val_lgbm, val_n1, val_mean])
stacker     = Ridge(alpha=1.0, positive=True, fit_intercept=False)
stacker.fit(preds_stack, y_val.values)
print(f"\nPoids Ridge : LightGBM={stacker.coef_[0]:.3f} | N-1={stacker.coef_[1]:.3f} | Mean={stacker.coef_[2]:.3f}")

val_blend = np.clip(np.round(stacker.predict(preds_stack)), 0, None).astype(int)
print(f"WAPE stacking final        : {wape_score(val_blend, y_val.values):.4f}")

print()
print("Benchmarks :")
print("  Baseline N-1     → ~1.387")
print("  Blend N-1 + mean → ~1.259")
print("  Ancien modèle    → 1.1029")
print("  LR dynamique     → ~0.85")

# Feature importance
feat_imp = pd.DataFrame({
    "feature":    FEATURES,
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)
display(feat_imp.head(20))

### 6. Génération & Sauvegarde des Prédictions

In [0]:
# COMMAND ----------
test_preds_raw = model.predict(X_test, num_iteration=model.best_iteration)
test_preds     = np.clip(np.round(test_preds_raw), 0, None).astype(int)

test_rows = all_df.loc[test_mask, ["semaine", "code_agence", "code_article"]].copy()
test_rows["quantite"] = test_preds

# Post-processing zéros
test_rows = test_rows.merge(pair_zero_rate, on=["code_agence","code_article"], how="left")
test_rows["quantite"] = np.where(test_rows["zero_rate_global"] > 0.98, 0, test_rows["quantite"])

# Blend via join
test_rows = test_rows.rename(columns={"semaine": "semaine_target"})
test_rows = test_rows.merge(
    n1_lookup, on=["semaine_target","code_agence","code_article"], how="left"
)
test_rows = test_rows.rename(columns={"semaine_target": "semaine"})

test_lgbm = test_rows["quantite"].values.astype(float)
test_n1   = test_rows["quantite_n1"].fillna(0).values.astype(float)

test_rows2 = all_df.loc[test_mask, ["semaine","code_agence","code_article"]].copy()
test_rows2 = test_rows2.merge(pair_mean_lookup, on=["code_agence","code_article"], how="left")
test_mean  = test_rows2["pair_mean_bl"].fillna(0).values.astype(float)

preds_stack_test = np.column_stack([test_lgbm, test_n1, test_mean])
test_blend       = np.clip(np.round(stacker.predict(preds_stack_test)), 0, None).astype(int)

test_rows["quantite"] = test_blend
print(f"Prédictions générées : {len(test_rows):,} (attendu : 272 344)")

predictions_spark = (
    spark.createDataFrame(test_rows[["semaine","code_agence","code_article","quantite"]])
    .withColumn("code_agence",  F.col("code_agence").cast(LongType()))
    .withColumn("code_article", F.col("code_article").cast(LongType()))
    .withColumn("quantite",     F.col("quantite").cast(LongType()))
)
predictions_spark.write.mode("overwrite").saveAsTable(TABLE_PREDICTIONS)
print(f"✅ Sauvegardé dans : {TABLE_PREDICTIONS}")